# Salon Experiment & Causal Analysis\n\nThis notebook follows the v2.0 design document. It is a scaffold: update file paths and column mappings after the real data are supplied. Do not invent or report effects from placeholder data.

## 0. Imports and reproducibility

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_quality import sample_ratio_mismatch, standardised_mean_difference
from src.experiment import retention_difference_ci, welch_revenue_test, client_bootstrap_mean_difference
from src.cuped import apply_cuped

SEED = 2026
np.random.seed(SEED)
DATA_DIR = PROJECT_ROOT / "data"


## 1. Load data (configure after receiving the dataset)

In [ ]:
# Replace these paths with the actual supplied files.
# appointments = pd.read_csv(DATA_DIR / "raw" / "appointments.csv")
# services = pd.read_csv(DATA_DIR / "raw" / "services.csv")
# stylists = pd.read_csv(DATA_DIR / "raw" / "stylists.csv")
# product_sales = pd.read_csv(DATA_DIR / "raw" / "product_sales.csv")
# assignment = pd.read_csv(DATA_DIR / "raw" / "experiment_assignment.csv")

# Stop rather than silently continue with fabricated data.
raise FileNotFoundError(
    "Add the authorised source datasets under data/raw/ and configure the file paths above."
)


## 2. Schema, row-grain and key validation\n\nCheck row counts, date ranges, duplicate primary keys, orphan foreign keys and missingness. Keep an exclusion log for every cleaning rule.

In [ ]:
# Example checks once dataframes are loaded:
# assert appointments["appointment_id"].notna().all()
# print("Rows:", len(appointments))
# print("Duplicate appointment IDs:", appointments["appointment_id"].duplicated().sum())
# display(appointments.isna().mean().sort_values(ascending=False).head(20))


## 3. Recover no-shows and map service taxonomy\n\nApply the exact rule in the design document. Validate recovered no-shows against a manually reviewed 300-record sample. Keep disputed records flagged for sensitivity analysis. Map service names with RapidFuzz score bands: >=90 auto-map, 70–89 manual review, <70 unmapped.

In [ ]:
# TODO: implement against the source-specific status and cancellation fields.
# Do not classify every cancellation as a no-show.


## 4. Build visit-level and client-level analytical tables\n\nA visit groups service lines for the same client and outlet within a 30-minute window. Keep anonymous walk-ins for outlet-level revenue/capacity only; exclude them from client-level retention.

In [ ]:
# TODO: build visits and client outcomes.
# Reconcile net revenue to POS within +/-0.5%.
# Exclude clients whose 90-day outcome window is immature; count them in the QA log.


## 5. Assignment validation and SRM (run before exclusions)

In [ ]:
# assignment must contain one row per randomised client and treatment in {0, 1}.
# assert assignment["client_id"].is_unique
# assert set(assignment["treatment"].dropna().unique()).issubset({0, 1})
# srm = sample_ratio_mismatch(assignment)
# print(srm)
# assert srm["p_value"] > 0.001, "SRM red flag: investigate assignment pipeline before outcome analysis."


## 6. Pre-treatment balance

In [ ]:
# For each pre-treatment covariate, calculate SMD between arms.
# Use |SMD| <= 0.10 as the practical threshold and report the maximum.
# Include prior 90-day visits / retention history as a key covariate.


## 7. ITT effect estimates and confidence intervals

In [ ]:
# client_level must contain one row per randomised eligible client with a mature outcome.
# retention_result = retention_difference_ci(client_level)
# revenue_result = welch_revenue_test(client_level)
# revenue_bootstrap = client_bootstrap_mean_difference(
#     client_level, outcome="revenue_per_visit", n_boot=10_000, seed=SEED
# )
# print(retention_result, revenue_result, revenue_bootstrap)


## 8. Guardrails

In [ ]:
# Evaluate contribution margin (non-inferiority margin -5%),
# complaints/refunds (+2 per 1,000 visits), capacity / booking lead-time,
# and opt-out rate (hard ceiling 1.5%). Guardrails are evaluated regardless
# of whether the primary outcome passes.


## 9. Historical Difference-in-Differences and event study

In [ ]:
# Build outlet-month panel with >=12 months pre-period and post-period.
# Fit outlet and month fixed effects with treated_outlet x post.
# Check parallel trends with event-study leads, slope comparison, placebo dates
# and Honest DiD sensitivity. If rollout is staggered, use a heterogeneity-robust
# group-time estimator rather than naive TWFE.


## 10. Segment interactions and multiple testing

In [ ]:
# Pre-specify outlet tier, service category, new/returning status,
# baseline frequency, spend tier and stylist level.
# Use treatment x segment interactions; report CIs and apply BH FDR at q=0.10.


## 11. CUPED / regression adjustment

In [ ]:
# Example after client_level is ready:
# adjusted, cuped_info = apply_cuped(client_level, "retained_90d", "pre_90d_visits")
# print(cuped_info)
# Compare unadjusted and adjusted effect estimates, CI widths, required sample
# and calendar-time savings. Covariates must be measured before randomisation.


## 12. Decision memo inputs\n\nApply the pre-registered rule: **GO** only if the lower 95% CI bound on retention exceeds +2.0 pp, delivery is >90%, SRM/balance pass and all guardrails hold. **NO-GO** if the upper bound is below +2.0 pp, the estimate is negative, or a hard guardrail breaks. Otherwise **EXTEND**. Do not write a decision until the actual data and assumption tests are complete.